In [ ]:
import streamlit as st
import pandas as pd

st.title("Konut Kredisi Hesaplama Modülü")
st.caption("Gayrimenkul alımında kullanılacak banka konut kredisi ödeme planı simülasyonu.")

st.markdown("""
<div class="content-card">
    <h4>Finansman ve Kredi Parametreleri</h4>
    Alınacak konut bedeli, kullanılacak peşinat oranı ve tercih edilen aylık faiz oranını giriniz.
</div>
""", unsafe_allow_html=True)

banka_seçimi = st.selectbox(
    "Banka Oran Referansı Seçiniz:",
    ["Ziraat / Halkbank / Vakıfbank (Kamu) - %2.79", "Özel Banka Ortalaması - %2.89", "Özel Faiz Oranı Girişi"],
    key="banka_secim"
)

col1, col2 = st.columns(2)
with col1:
    tutar = st.number_input("Konut Rayiç Bedeli (TL)", value=5_000_000.0, step=100000.0, key="kredi_page_tutar")
    pesinat = st.slider("Özkaynak / Peşinat Oranı (%)", 10, 90, 20, 5, key="kredi_page_pesinat")
with col2:
    vade = st.selectbox("Geri Ödeme Vadesi (Ay)", [36, 48, 60, 84, 120, 180, 240], index=4, key="kredi_page_vade")
    if banka_seçimi == "Özel Faiz Oranı Girişi":
        faiz = st.number_input("Aylık Akdi Faiz Oranı (%)", 0.1, 10.0, 2.79, 0.01, key="kredi_page_faiz")
    elif "Kamu" in banka_seçimi:
        faiz = 2.79
        st.info("Kamu bankası referans aylık faiz oranı: %2.79")
    else:
        faiz = 2.89
        st.info("Özel banka referans aylık faiz oranı: %2.89")

pesinat_tutar = tutar * (pesinat / 100)
kredi = tutar - pesinat_tutar

if kredi > 0:
    r = faiz / 100
    taksit = (kredi * r * (1 + r)**vade) / ((1 + r)**vade - 1)
    toplam = taksit * vade
    
    st.divider()
    m1, m2, m3 = st.columns(3)
    m1.metric("Peşinat Tutarı", f"{pesinat_tutar:,.0f} TL")
    m2.metric("Kredi Tutarı", f"{kredi:,.0f} TL")
    m3.metric("Aylık Taksit Bedeli", f"{taksit:,.2f} TL")
    
    st.markdown(f"""
    <div class="rapor-kart">
        <h4>Geri Ödeme Planı Finansal Özeti</h4>
        <ul>
            <li><strong>Ana Para Tutarı:</strong> {kredi:,.2f} TL</li>
            <li><strong>Toplam Tahakkuk Eden Faiz:</strong> {toplam - kredi:,.2f} TL</li>
            <li><strong>Toplam Geri Ödeme Bedeli:</strong> {toplam:,.2f} TL</li>
        </ul>
    </div>
    """, unsafe_allow_html=True)

    st.divider()
    st.markdown("### Örnek Yıllık Ödeme Özet Tablosu")
    
    yillik_ozet = []
    kalan_anapara = kredi
    for ay in range(1, vade + 1):
        faiz_payi = kalan_anapara * r
        anapara_payi = taksit - faiz_payi
        kalan_anapara -= anapara_payi
        if ay % 12 == 0 or ay == vade:
            yillik_ozet.append({
                "Taksit (Ay)": f"{ay}. Ay",
                "Aylık Taksit": f"{taksit:,.2f} TL",
                "Kalan Anapara": f"{max(0, kalan_anapara):,.2f} TL"
            })
    
    st.dataframe(pd.DataFrame(yillik_ozet), use_container_width=True, hide_index=True)